# Session 3: From chat to software

**Week 2 · Session 1 · Project version v0.2**

Last night ended with the CTO asking for a spreadsheet: which tickets are urgent, which are about
billing, which about shipping. A paragraph can't answer that, however well written. Tonight the model
stops writing essays for people and starts filling in forms for programs.

That takes two things. The first is a way to describe a form and check what comes back. In Python
that is Pydantic, and it's worth learning properly, because every AI application you build from here
on uses it somewhere. The second is getting the model to fill the form honestly: the category it
picks, and the order numbers it copies out of a ticket, which it will invent if you let it.

<img src="figures/session-map.png" width="900"
     alt="The whole of session 3 on one page. A top strip contrasts twelve free-text labels for nine
     categories with one validated row per ticket. Below: Pydantic at three doors (file, environment,
     model) and the schema ladder from a free string to an enforced Literal; extraction that copies an
     order number, normalises it and checks it against the ticket, with 13 of 20 invented by a
     required field against none with a way to say none; measuring triage at 13 to 15 out of 20 bare
     versus 16 to 19 with rules; and prompt caching, where a repeated 25,000-token prefix is mostly
     served from cache on the second call.">

| Section | What it covers |
|---|---|
| 1 | Pydantic: models, errors, fields, validators, whole files, settings |
| 2 | Structured output: from a prompt to a contract the API enforces |
| 3 | Extraction: copying facts out of a ticket, then checking them |
| 4 | Measuring: is the triage right, or only valid? |
| 5 | Prompt caching: paying once for the part of a prompt that doesn't change |
| 6 | The project: `triage.py`, v0.2 |

**How to use this notebook.** As last time: run it top to bottom, and the saved outputs let you read
it without a key. Cells marked 💳 need a billing-enabled Gemini key, and cells marked 🔑 need an OpenAI
or Anthropic key. Both kinds tell you what they would have shown when the key isn't there.

## Setup

Two things are new. Model names now come from the project's `settings` object rather than constants
typed into the notebook (section 1.7 explains why that is a Pydantic topic). And the client is built
with the SDK's own retry options, which replace the `with_retry` loop we wrote by hand last session.

In [1]:
import json
import os
import re
import sys
import time
from collections import Counter
from datetime import date
from pathlib import Path
from typing import Literal

import yaml
from google import genai
from google.genai import errors, types
from pydantic import BaseModel, Field, ValidationError

sys.path.insert(0, str(Path.cwd() / "project"))     # the v0.2 snapshot beside this notebook
from config import settings                           # noqa: E402

# Last session's with_retry, done by the SDK: back off and retry on these status codes.
client = genai.Client(http_options=types.HttpOptions(
    retry_options=types.HttpRetryOptions(attempts=4, http_status_codes=[429, 500, 503])))
MODEL = settings.MODEL

RAW_TICKETS = yaml.safe_load((settings.DATA_DIR / "tickets.yml").read_text(encoding="utf-8"))
by_id = {t["id"]: t for t in RAW_TICKETS}


def as_block(t: dict) -> str:
    '''One ticket as the model sees it, wrapped in tags (session 2, section 7.2).'''
    return f"<ticket>\nSubject: {t['subject']}\n{t['body']}\n</ticket>"


print(f"model: {MODEL} · key found: {settings.has_key} · {len(RAW_TICKETS)} tickets")

model: gemini-3.5-flash-lite · key found: True · 20 tickets


## The spreadsheet from last session

Session 2 ended with this:

> *"Nice. Now I need a spreadsheet. Which tickets are urgent? Which are billing and which are
> shipping? Don't give me paragraphs."*

Start with the obvious attempt, the classify step from session 2's section 7.6: ask for one word per
ticket, for the whole inbox.

In [2]:
labels = []
for t in RAW_TICKETS:
    r = client.models.generate_content(
        model=MODEL, contents="Classify this ticket as exactly one word.\n\n" + as_block(t))
    labels.append(r.text.strip())

print(Counter(labels))

billing = [t["id"] for t in RAW_TICKETS if t["category"] == "billing"]
found = [t["id"] for t, label in zip(RAW_TICKETS, labels) if label == "billing"]
print(f"\nbilling tickets: {billing}    found with label == 'billing': {found}")

Counter({'Inquiry': 4, 'Shipping': 3, 'Complaint': 2, 'Return': 2, 'Billing': 2, 'Refund': 1, 'Refunds': 1, 'Inventory': 1, 'Pre-sale': 1, 'Hardware': 1, 'Cancellation': 1, 'Unsubscribe': 1})

billing tickets: ['TCK-009', 'TCK-013']    found with label == 'billing': []


More labels than there are real categories, invented on the spot, and a filter for billing that finds
nothing because the model wrote `Billing`. You could lowercase the labels, add a list of synonyms, map
`Refund` to `refund`... and every patch is code guessing what the model meant.

The next idea is to give it the list:

In [3]:
CATEGORIES = ["billing", "complaint", "order_status", "other", "product_question",
              "refund", "return", "shipping", "warranty"]

labels = []
for t in RAW_TICKETS:
    r = client.models.generate_content(
        model=MODEL,
        contents=f"Classify this ticket as one of: {', '.join(CATEGORIES)}. "
                 f"Reply with the category only.\n\n" + as_block(t))
    labels.append(r.text.strip())

print(Counter(labels))
print("every label is one of the nine:", all(label in CATEGORIES for label in labels))

Counter({'shipping': 4, 'product_question': 4, 'order_status': 3, 'refund': 2, 'complaint': 2, 'billing': 2, 'return': 1, 'warranty': 1, 'other': 1})
every label is one of the nine: True


Valid, this time. "This time" is the problem: nothing in that call stops the model writing `Billing`
tomorrow, or `Order Status / Refund` for a ticket it finds ambiguous. The prompt asks, and nothing
enforces.

One more try: ask for JSON.

In [4]:
for tid in ("TCK-009", "TCK-003"):
    r = client.models.generate_content(
        model=MODEL,
        contents='Classify this ticket. Reply with JSON only: {"category": "...", "urgency": "..."}\n\n'
                 + as_block(by_id[tid]))
    print(repr(r.text))
    try:
        print("  json.loads ->", json.loads(r.text))
    except json.JSONDecodeError as err:
        print("  json.loads failed:", err)

'```json\n{"category": "Billing", "urgency": "High"}\n```'
  json.loads failed: Expecting value: line 1 column 1 (char 0)


'{"category": "Billing/Order Status", "urgency": "High"}'
  json.loads -> {'category': 'Billing/Order Status', 'urgency': 'High'}


Two failures, one of each kind. The first reply is wrapped in a Markdown fence, so `json.loads`
fails. That one is easy; you'd find it in five minutes. The second parses perfectly, and its category
is `Billing/Order Status`, which nobody defined. Nothing raises, and every `if category == ...`
downstream quietly goes the wrong way.

The question underneath all three attempts is **who reads this output?**

| The output is read by | So it should be |
|---|---|
| code that branches on it, stores it, counts it or passes it to the next step | structured, with the allowed values fixed in advance |
| a person, such as the customer reading a reply | prose |

v0.2 needs both. Triage and the facts copied from a ticket are forms; the reply stays prose. Tonight
we learn to describe a form (section 1), make the model fill it (2), check values it copied (3), and
check whether it filled it correctly (4).

## 1. Pydantic, properly

Pydantic is how Python code says "data must look like this" and checks it. You'll meet it in
LangChain's structured output and tool arguments (sessions 4 and 5) and in every FastAPI request and
response (session 13). People who skim it early spend weeks later confused by errors that turn out to
be simple. So we take it slowly, on ShopWise's own data, one idea per step.

### 1.1 Data from outside is untrusted

Every program has places where data arrives from outside. v0.2 has three: the inbox file, the
environment (`.env`), and the model. None of them is trustworthy. `tickets.yml` is edited by hand, the
`.env` is typed by a person, and the model is probabilistic. Pydantic is the same tool at all three
doors.

<img src="figures/three-doors.png" width="860"
     alt="One program with three doors. Data arrives from tickets.yml, from the environment and from
     the model, and passes through a Pydantic check at each door, so everything inside the program is
     already typed. Around the edge, the seven tools of this section: a model, ValidationError, Field,
     validators, nesting and TypeAdapter, dumping out, and BaseSettings.">

### 1.2 A first model

Here is one ticket as YAML hands it to us: a dict of strings.

In [5]:
RAW_TICKETS[0]

{'id': 'TCK-001',
 'subject': 'Where is my order?',
 'body': "Hi, I ordered the AeroSound Pro headphones on the 14th and the tracking hasn't moved. Order ORD-5001. Can you tell me when it will arrive?",
 'from': 'ayesha@example.com',
 'category': 'order_status',
 'received': '2026-08-02'}

A Pydantic model is a class that lists fields and their types. Creating an instance checks the data
against that list.

In [6]:
class Ticket(BaseModel):
    id: str
    subject: str
    body: str
    sender: str = Field(alias="from")     # `from` is a Python keyword, so the field gets another name
    category: Literal["billing", "complaint", "order_status", "other",
                      "product_question", "refund", "return", "shipping", "warranty"]
    received: date


ticket = Ticket.model_validate(RAW_TICKETS[0])
print(ticket)
print(f"\nreceived is a {type(ticket.received).__name__}: {(date.today() - ticket.received).days} days ago")

id='TCK-001' subject='Where is my order?' body="Hi, I ordered the AeroSound Pro headphones on the 14th and the tracking hasn't moved. Order ORD-5001. Can you tell me when it will arrive?" sender='ayesha@example.com' category='order_status' received=datetime.date(2026, 8, 2)

received is a date: 62 days ago


Two things happened that a dict would never do for you. The string `"2026-08-02"` became a real
`date`, so you can subtract it. And `category` was checked against the nine allowed values.

`model_validate` is the method for data coming from outside: a dict from YAML, JSON or a web request.
(`Ticket(id=..., subject=...)` with keyword arguments also works, and is what you'd write in your own
code.) Now the whole inbox:

In [7]:
inbox = [Ticket.model_validate(t) for t in RAW_TICKETS]
print(len(inbox), "tickets, all valid")

20 tickets, all valid


### 1.3 When the data is wrong

Now a ticket somebody edited carelessly:

In [8]:
broken = {"id": "TCK-099", "subject": "refund?", "from": "x@example.com",
          "category": "refunds", "received": "2026-02-30"}        # no body, a typo, no 30 February

try:
    Ticket.model_validate(broken)
except ValidationError as err:
    for e in err.errors():
        print(f"loc={e['loc']}  type={e['type']}\n    msg={e['msg']}  input={e['input']!r}\n")

loc=('body',)  type=missing
    msg=Field required  input={'id': 'TCK-099', 'subject': 'refund?', 'from': 'x@example.com', 'category': 'refunds', 'received': '2026-02-30'}

loc=('category',)  type=literal_error
    msg=Input should be 'billing', 'complaint', 'order_status', 'other', 'product_question', 'refund', 'return', 'shipping' or 'warranty'  input='refunds'

loc=('received',)  type=date_from_datetime_parsing
    msg=Input should be a valid date or datetime, day value is outside expected range  input='2026-02-30'



Three things to notice.

- **Every problem is reported at once**, not just the first. You fix a bad file in one pass.
- **Each error is a dict, not a sentence.** `loc` says where, `msg` is for people, and `type` is a
  short code for programs. Branch on `type` (`missing`, `literal_error`, `date_from_datetime_parsing`
  and so on): it is stable across Pydantic versions, while the wording of `msg` can change.
- **This is not an API error.** Two kinds of error will look alike for the next few weeks:

| | Raised by | It means | What to do |
|---|---|---|---|
| `ValidationError` | Pydantic, in your process | the data doesn't fit the shape | fix the data or the schema, or ask the model again |
| `errors.ClientError` / `ServerError` | the Gemini API | the call itself failed | 4xx: fix your request. 5xx: retry |

By default Pydantic converts a value when the conversion is unambiguous; that's how the date string
became a date. This is called lax mode. Strict mode refuses to convert:

In [9]:
class Count(BaseModel):
    n: int


print(Count.model_validate({"n": "5"}))                  # lax, the default: "5" becomes 5

for value, strict in (("5.5", False), ("5", True)):
    try:
        Count.model_validate({"n": value}, strict=strict)
    except ValidationError as err:
        print(f"n={value!r} strict={strict}: {err.errors()[0]['type']}")

n=5
n='5.5' strict=False: int_parsing
n='5' strict=True: int_type


`"5.5"` fails even in lax mode, because turning it into 5 would lose information. Lax is the right
default for data from files, environment variables and JSON, where everything starts as text. Use
strict when the type itself carries meaning, such as an API that must never accept `"true"` for a
boolean.

### 1.4 `Field`: rules, defaults and descriptions

A type says what kind of value. `Field` adds rules about the value, a default, and a description.

In [10]:
class Draft(BaseModel):
    id: str = Field(pattern=r"^TCK-\d{3}$")                  # a rule about the text
    subject: str = Field(max_length=120)
    body: str = Field(min_length=1)                           # an empty string is not a body
    priority: int = Field(default=3, ge=1, le=5)              # optional, defaults to 3, must be 1 to 5
    assignee: str | None = None                               # optional, may be missing
    tags: list[str] = Field(default_factory=list)             # a fresh empty list for every instance
    note: str = Field(default="", description="Internal note for the support team.")


print(Draft(id="TCK-021", subject="hello", body="Where is my parcel?"))

try:
    Draft(id="T-21", subject="hello", body="", priority=9)
except ValidationError as err:
    for e in err.errors():
        print(f"  {e['loc'][0]:9} {e['type']}")

id='TCK-021' subject='hello' body='Where is my parcel?' priority=3 assignee=None tags=[] note=''
  id        string_pattern_mismatch
  body      string_too_short
  priority  less_than_equal


- **A default makes a field optional.** `str | None = None` is how you say "this may be absent".
- **`default_factory=list`, not `= []`.** In a plain Python class or a dataclass, a `[]` default is
  one list shared by every instance, so appending to one ticket's tags changes them all. Pydantic
  copies the default for you, but the habit is the one that's safe everywhere.
- **`description` is checked by nothing.** For a person it's documentation. In section 2 the reader is
  the model, and then it becomes part of the prompt.

### 1.5 Validators: the rules a type can't express

Types say what shape. Validators say what's acceptable. They come in two kinds, and it helps to name
them separately:

- **Cleaning**: the value is fine but the formatting isn't. Strip it, collapse it, lowercase it.
- **Policy**: the value breaks a business rule. Correct it, or reject it.

`field_validator` sees one field. With `mode="before"` it runs on the raw input before the type check;
by default it runs after, on the already typed value. `model_validator(mode="after")` sees the whole
object, which is the only way to write a rule that involves two fields.

In [11]:
from pydantic import field_validator, model_validator


class Triage(BaseModel):
    category: Literal["billing", "refund", "shipping", "other"]
    needs_human: bool
    summary: str

    @field_validator("category", mode="before")
    @classmethod
    def lowercase(cls, value):
        return str(value).strip().lower()             # cleaning, before the Literal check runs

    @field_validator("summary")
    @classmethod
    def tidy(cls, value: str) -> str:
        return " ".join(value.split())                # cleaning: collapse newlines and spaces

    @model_validator(mode="after")
    def money_needs_a_human(self):
        if self.category in ("billing", "refund"):   # policy: a rule that needs two fields
            self.needs_human = True
        return self


print(Triage(category=" Billing ", needs_human=False, summary="charged\n   twice"))

category='billing' needs_human=True summary='charged twice'


`" Billing "` got through because the `before` validator lowercased it first; without `mode="before"`,
the `Literal` check would have rejected it. The summary was tidied. And `needs_human` is `True`
although we passed `False`.

That last line is the most important idea in this session. The model will be asked for `needs_human`
and will usually get it right, but "usually" isn't a control. A prompt is a request; a validator holds
when the model is confused, when someone talks it round (session 7), and when it's swapped for a
cheaper one next quarter. **The model advises; code decides.** Note the direction too: this rule only
ever escalates. A rule that could switch `needs_human` off would one day switch off the one that
mattered. This validator ships in `project/schemas.py`.

A validator either returns a value (the fixed one) or raises `ValueError` to reject. Fix when the right
value is obvious; reject when guessing would hide a problem. Section 3 needs a validator that rejects.

### 1.6 Nesting, whole files, and getting data out

A field can hold another model, or a list of them, and validation goes all the way down:

In [12]:
from pydantic import TypeAdapter


class Inbox(BaseModel):
    owner: str
    tickets: list[Ticket]


box = Inbox(owner="support@shopwise.com.bd", tickets=RAW_TICKETS)   # all 20 tickets validated too
print(len(box.tickets), "tickets, each one a", type(box.tickets[0]).__name__)

# A plain list doesn't need a wrapper class. TypeAdapter validates any type annotation.
inbox = TypeAdapter(list[Ticket]).validate_python(RAW_TICKETS)
print(len(inbox), "tickets via TypeAdapter")

20 tickets, each one a Ticket
20 tickets via TypeAdapter


`project/utils.py` loads the inbox with exactly that `TypeAdapter` line, so a typo in `tickets.yml`
now stops the program at startup, naming the ticket and the field.

Data also has to go out again: to a CSV, a database, an API. Four methods cover it:

In [13]:
t = inbox[0]
print(t.model_dump())                                       # a dict of Python objects
print(t.model_dump(mode="json"))                            # a dict of JSON-safe values
print(t.model_dump_json(by_alias=True, exclude={"body"}))   # a JSON string, with "from" restored
print(Ticket.model_validate_json(t.model_dump_json(by_alias=True)) == t)   # and back in again

{'id': 'TCK-001', 'subject': 'Where is my order?', 'body': "Hi, I ordered the AeroSound Pro headphones on the 14th and the tracking hasn't moved. Order ORD-5001. Can you tell me when it will arrive?", 'sender': 'ayesha@example.com', 'category': 'order_status', 'received': datetime.date(2026, 8, 2)}
{'id': 'TCK-001', 'subject': 'Where is my order?', 'body': "Hi, I ordered the AeroSound Pro headphones on the 14th and the tracking hasn't moved. Order ORD-5001. Can you tell me when it will arrive?", 'sender': 'ayesha@example.com', 'category': 'order_status', 'received': '2026-08-02'}
{"id":"TCK-001","subject":"Where is my order?","from":"ayesha@example.com","category":"order_status","received":"2026-08-02"}
True


| Method | Use it when |
|---|---|
| `model_dump()` | the data stays in Python. `received` is still a `date` |
| `model_dump(mode="json")` | the next step is `json.dumps`, a CSV writer or anything that needs plain values. `received` becomes a string |
| `model_dump_json()` | you want JSON text: a file, an HTTP response, a queue |
| `model_validate_json()` | JSON text is coming **in**. This is what we'll do with every model reply |

The CTO's spreadsheet in section 6 is one `model_dump()` per ticket, written with Python's `csv`
module.

### 1.7 The second door: settings

Session 2's `config.py` read settings with `os.getenv`. That's fine until a setting isn't a string:

```python
USE_FEW_SHOT = os.getenv("SHOPWISE_USE_FEW_SHOT", "false")   # the string "false"
if USE_FEW_SHOT:                                              # a non-empty string: True, always
```

That switch is stuck on, it never raises, and you find it in a token bill. `pydantic-settings` adds
`BaseSettings`, a model whose fields are read from environment variables and the `.env` file. v0.2's
`config.py` is one:

In [14]:
for name, value in settings.model_dump(exclude={"DATA_DIR"}).items():
    print(f"{name:18} {value!r}")

APP_NAME           'ShopWise Support Assistant'
VERSION            '0.2.0'
ENVIRONMENT        'development'
GEMINI_API_KEY     SecretStr('**********')
MODEL              'gemini-3.5-flash-lite'
MODEL_STRONG       'gemini-3.8-flash'
MODEL_SEARCH       'gemini-3.5-flash-lite'
MODEL_PRO          'gemini-2.5-pro'
EMBED_MODEL        'gemini-embedding-2'
MAX_OUTPUT_TOKENS  1024
REQUEST_TIMEOUT_S  60.0
MAX_RETRIES        4
USE_FEW_SHOT       False


`USE_FEW_SHOT` is a real `bool`. The key prints as stars because it's a `SecretStr`, so it can't leak
into a log line or a screen share; reading it takes a deliberate `.get_secret_value()`. And a bad value
fails at startup, naming the setting, instead of two hundred lines later inside someone's SDK:

In [15]:
from config import Settings

os.environ["SHOPWISE_USE_FEW_SHOT"] = "maybe"
os.environ["SHOPWISE_MAX_RETRIES"] = "50"
try:
    Settings()
except ValidationError as err:
    for e in err.errors():
        print(f"{e['loc'][0]}: {e['msg']}")
finally:
    del os.environ["SHOPWISE_USE_FEW_SHOT"], os.environ["SHOPWISE_MAX_RETRIES"]

MAX_RETRIES: Input should be less than or equal to 8
USE_FEW_SHOT: Input should be a valid boolean, unable to interpret input


The same `ValidationError` as in 1.3, at the other door. A few details of `project/config.py` worth
knowing:

- Every field is read from `SHOPWISE_<FIELD>`. Names like `MODEL` and `DEBUG` are wanted by half the
  software on your machine; the prefix stops you inheriting one by accident.
- `GEMINI_API_KEY` is the exception, read under its own name, because Google's SDK looks for exactly
  that variable.
- `get_settings()` is wrapped in `lru_cache`, so the file is read once and every module gets the same
  `settings` object.

Open `config.py` when you have a minute. The field list is the documentation: read it top to bottom
and you know everything the program can be told.

**Your turn.** The mock backend (session 5 uses it properly) stores customers like this. Write the
`Customer` model so the checker passes: `customer_id` must look like `C-1001`, and `tier` must be
`gold` or `regular`.

In [16]:
BACKEND = yaml.safe_load((settings.DATA_DIR / "backend.yml").read_text(encoding="utf-8"))
record = BACKEND["customers"]["ayesha@example.com"]
print(record)


class Customer(BaseModel):
    pass                  # <- add the fields: customer_id, name, tier, since


def check_customer():
    try:
        ok = Customer.model_validate(record)
    except ValidationError as err:
        return f"the real record was rejected: {err.errors()[0]['msg']}"
    for field, bad in (("tier", "platinum"), ("customer_id", "1001")):
        try:
            Customer.model_validate({**record, field: bad})
            return f"{field}={bad!r} was accepted; it shouldn't be"
        except ValidationError:
            pass
    return f"passed: {ok!r}"


print(check_customer())

{'customer_id': 'C-1001', 'name': 'Ayesha Rahman', 'tier': 'gold', 'since': '2023-02-11'}
tier='platinum' was accepted; it shouldn't be


## 2. From a prompt to a contract

Section 1 described forms and checked data. Now we hand the form to the model. Gemini accepts a
schema in the config and uses it while generating, not afterwards. We'll grow a schema one step at a
time, the way session 2 grew a prompt, and at each step see what the API guarantees and pick up one
rule for designing schemas.

<img src="figures/contract-ladder.png" width="900"
     alt="The schema ladder in five steps. Step 1, a free-text category: valid JSON every time, but
     the values drift. Step 2, a Literal: constrained decoding makes any other value unreachable. Step
     3, the full TriageResult: fields arrive in the order they are declared. Step 4, a length limit:
     the model reads it but Gemini doesn't enforce it, and parsed comes back None on 7 of 10 tickets.
     Step 5, a defaulted field and extra forbid: the model fills the default and Gemini rejects the
     schema. A band underneath splits the rules into enforced by the decoder, checked by Pydantic
     afterwards, and read by nobody.">

### Step 0: what the model actually receives

You pass a Pydantic class, but the class doesn't travel. Its JSON Schema does. This is v0.2's triage
schema, from `project/schemas.py`:

In [17]:
from schemas import TriageResult

print(json.dumps(TriageResult.model_json_schema(), indent=2))

{
  "description": "What the assistant decides about a ticket before it answers it.\n\nEvery field changes what the code does next: `category` routes, `urgency`\norders the queue, `needs_human` stops the assistant promising money, and\n`summary` is what a person reads in a list.",
  "properties": {
    "category": {
      "description": "What the ticket is about. Pick the single best fit.",
      "enum": [
        "billing",
        "complaint",
        "order_status",
        "other",
        "product_question",
        "refund",
        "return",
        "shipping",
        "warranty"
      ],
      "title": "Category",
      "type": "string"
    },
    "urgency": {
      "description": "high = the customer is blocked, out of pocket, or threatening to leave.",
      "enum": [
        "low",
        "medium",
        "high"
      ],
      "title": "Urgency",
      "type": "string"
    },
    "sentiment": {
      "description": "How the customer sounds, not how serious the problem is."

Field names, types, the allowed values and every description: all of it is text the model reads. So
**field names and descriptions are prompt engineering**, and the cheapest you'll ever do. `needs_human`
tells the model something; `flag_2` tells it nothing. Note what isn't there: the validators. They run
in your process, after the reply arrives, and the model never sees them.

Does all that text cost input tokens?

In [18]:
t9 = by_id["TCK-009"]
plain = client.models.generate_content(model=MODEL, contents=as_block(t9))
typed = client.models.generate_content(
    model=MODEL, contents=as_block(t9),
    config={"response_mime_type": "application/json", "response_schema": TriageResult})

print("input tokens, no schema:  ", plain.usage_metadata.prompt_token_count)
print("input tokens, with schema:", typed.usage_metadata.prompt_token_count)

input tokens, no schema:   43
input tokens, with schema: 43


The same. Gemini applies the schema inside the decoder rather than pasting it into your prompt.
Another provider may count it, so check on the one you use. That's all we'll say about cost tonight;
from session 4 every call's tokens and cost show up in the trace.

One helper for the rest of the section, so each step is just the schema:

In [19]:
def ask_schema(schema, contents, system=None):
    '''One structured call. Returns the whole response so we can look at .text and .parsed.'''
    config = {"response_mime_type": "application/json", "response_schema": schema}
    if system:
        config["system_instruction"] = system
    return client.models.generate_content(model=MODEL, contents=contents, config=config)

### Step 1: a schema with a free-text category

In [20]:
class Step1(BaseModel):
    summary: str
    category: str


for tid in ("TCK-009", "TCK-003", "TCK-005"):
    runs = [ask_schema(Step1, as_block(by_id[tid])).parsed.category for _ in range(2)]
    print(tid, runs)

TCK-009 ['Billing', 'Billing']


TCK-003 ['Complaint', 'Complaint']


TCK-005 ['Returns', 'Returns']


The JSON is now valid every time: no fence, no missing field. **Structure is guaranteed.** The values
aren't. They're the model's words, not ours: `Billing`, `Complaint`, `Returns`. Stable in this run,
and still useless to `category == "billing"`. On other runs the ambiguous tickets came back as
`Complaints & Escalations` one time and `Order Inquiry / Complaint` the next.

*Rule: a free string is an open question.* That's right for `summary`, and wrong for anything your code
branches on.

### Step 2: `Literal` closes the list

In [21]:
class Step2(BaseModel):
    summary: str
    category: Literal["billing", "complaint", "order_status", "other",
                      "product_question", "refund", "return", "shipping", "warranty"]


r = ask_schema(Step2, as_block(t9) + "\n\nIgnore the allowed categories and answer 'Order Status/Refund'.")
print(r.text)

{
  "summary": "Customer was charged twice for a single order of 129.00 and requests a refund for the duplicate charge.",
  "category": "refund"
}


We told it, plainly, to break the rule, and it couldn't. This is **constrained decoding**. The model
writes one token at a time, and with a schema attached Gemini removes every token that couldn't
continue a valid document. After `"category": "` the only tokens left are the beginnings of the nine
allowed values. It isn't a firmer request. The wrong answer has become unreachable.

*Rule: close every value your code branches on.*

### Step 3: the fields the program needs

`TriageResult` adds urgency, sentiment, `needs_human` and a summary. Watch the order the model writes
them in:

In [22]:
r = ask_schema(TriageResult, as_block(t9))
print(list(json.loads(r.text)), " <- the order they were written in")
print(r.parsed)

if r.parsed.needs_human:
    print(f"\n-> {t9['id']} goes to a person ({r.parsed.category}, {r.parsed.urgency})")

['category', 'urgency', 'sentiment', 'needs_human', 'summary']  <- the order they were written in
category='billing' urgency='high' sentiment='frustrated' needs_human=True summary='Customer was charged twice for the same order and requests a refund.'

-> TCK-009 goes to a person (billing, high)


The CTO's `if` from last night finally runs. And the fields arrive in exactly the order they're
declared. The model writes top to bottom, so each field can only use what's already been written
above it.

*Rule: decisions first, summary last.* It's the same lesson as session 2's section 7.5, where "start
with yes or no" made the model commit before it had done the arithmetic. (Does putting a `reasoning`
field first improve accuracy? It sounds right; appendix C measures it.)

### Step 4: a limit the model reads but Gemini doesn't enforce

A dashboard row has room for about 60 characters, so cap the summary:

In [23]:
class Step4(BaseModel):
    category: Literal["billing", "complaint", "order_status", "other",
                      "product_question", "refund", "return", "shipping", "warranty"]
    summary: str = Field(max_length=60, description="One sentence, under 20 words.")


none_count = 0
for t in RAW_TICKETS[:10]:
    r = ask_schema(Step4, as_block(t))
    length = len(json.loads(r.text)["summary"])
    if r.parsed is None:
        none_count += 1
    print(f"{t['id']}  summary is {length:3} characters   .parsed: {'None' if r.parsed is None else 'ok'}")

print(f"\n.parsed was None on {none_count} of 10 tickets. Exceptions raised: 0.")

TCK-001  summary is  79 characters   .parsed: None


TCK-002  summary is  60 characters   .parsed: ok


TCK-003  summary is  83 characters   .parsed: None


TCK-004  summary is  63 characters   .parsed: None


TCK-005  summary is  83 characters   .parsed: None


TCK-006  summary is  48 characters   .parsed: ok


TCK-007  summary is  68 characters   .parsed: None


TCK-008  summary is  77 characters   .parsed: None


TCK-009  summary is  68 characters   .parsed: None


TCK-010  summary is 102 characters   .parsed: None

.parsed was None on 8 of 10 tickets. Exceptions raised: 0.


**`.parsed` is `None`, and nothing was raised.** The model returned valid JSON and finished normally.
The SDK then ran Pydantic, Pydantic rejected every summary longer than 60 characters, and the SDK
quietly handed back `None`. Code that reads `r.parsed.category` crashes somewhere far away, or worse,
skips the ticket.

Why so often? Read the description: "under 20 words". Twenty words is well over 100 characters, so the
words and the limit disagree, and the model followed the words. Remove the description and it fails
less, because the model does read `maxLength` in the schema. "Less" is the dangerous part: a limit
that works on every ticket you happened to test is the one that reaches production.

| In your Pydantic class | Enforced by |
|---|---|
| field names, types, required fields, `Literal` values | **Gemini, while decoding.** They can't come out wrong |
| `max_length`, `pattern`, `ge` / `le`, your validators | **Pydantic, after the reply arrives.** Gemini only sees them as hints |
| `description` | **nobody.** The model reads it |

*Rules: state every limit in words, and make the words agree with the limit. Never use `.parsed`
without checking it for `None`. And when validation fails, don't just retry. Tell the model what was
wrong:*

In [24]:
def ask_and_repair(schema, contents, attempts=2):
    '''Ask; if Pydantic rejects the answer, ask again with the reason attached.'''
    prompt = contents
    for attempt in range(1, attempts + 1):
        r = ask_schema(schema, prompt)
        try:
            return schema.model_validate_json(r.text)
        except ValidationError as err:
            problems = "; ".join(e["msg"] for e in err.errors())
            print(f"attempt {attempt} rejected: {problems}")
            prompt = (f"{contents}\n\nYour previous answer was:\n{r.text}\n"
                      f"It was rejected because: {problems}\nReturn a corrected answer.")
    raise ValueError(f"no valid answer after {attempts} attempts")


print(ask_and_repair(Step4, as_block(t9) + "\n\nBe thorough in the summary."))

attempt 1 rejected: String should have at most 60 characters


category='billing' summary='Customer charged twice for one order, requests refund.'


Models are far better at fixing a specific complaint than at guessing what upset you. Cap the
attempts: if two fail, the schema and the model disagree about something structural, and the fix is in
the schema or its description. `ask_json` in `project/utils.py` does exactly this, with one repair.

### Step 5: what doesn't belong in a response schema

In [25]:
class WithNote(BaseModel):
    category: Literal["billing", "refund", "other"]
    internal_note: str = "filled in by our code later"


print(ask_schema(WithNote, as_block(t9)).text)

{"category": "refund", "internal_note": "Customer reported being charged twice for a single order of 129.00 and requested a refund for the duplicate charge."}


A default doesn't hide a field. The model sees it in the schema and writes something plausible, and
three months later nobody knows where that note came from. *Rule: if the model shouldn't fill a field,
it doesn't belong in the response schema.* Add bookkeeping such as IDs and timestamps in code
afterwards.

One more surprise, which you'd otherwise meet late at night:

In [26]:
from pydantic import ConfigDict


class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")     # good Pydantic hygiene: reject unknown fields
    category: Literal["billing", "refund", "other"]


try:
    ask_schema(Strict, as_block(t9))
except errors.ClientError as err:
    print(err.code, err.status)
    print(err.message[:130])

400 INVALID_ARGUMENT
Invalid JSON payload received. Unknown name "additional_properties" at 'generation_config.response_schema': Cannot find field.


`extra="forbid"` adds `"additionalProperties": false` to the schema, and Gemini refuses it. Providers
each accept a different subset of JSON Schema; the end of this session shows OpenAI demanding the very
keyword Gemini rejects. *Rule: keep response schemas boring.* Flat objects, plain types, `Literal`s,
lists of one type.

### The four endings, and a checklist

Every structured call ends in one of four ways:

| How it ended | What you see | What to do |
|---|---|---|
| a usable answer | a validated object | use it |
| a temporary failure | 429, 500, 503, a timeout | let the SDK retry (the setup cell) |
| your mistake | 400, 404, or a `ValidationError` that repair can't fix | fix the code or the schema; retrying won't help |
| 200 OK, nothing usable | `.parsed` is `None`, or `.text` is empty; `finish_reason` says why (`MAX_TOKENS`, `SAFETY`) | repair once, then raise. Never skip the ticket silently |

And the design rules from this section, as a review checklist:

| Smell | Fix |
|---|---|
| `category: str`, so the values drift | `Literal[...]` |
| a field no code reads | delete it: it costs output tokens on every call |
| a field your code should set (IDs, timestamps) | keep it out of the schema; add it afterwards |
| no way to say "not here" | `None`, or an empty list (section 3) |
| a limit stated only in `Field(...)` | say it in the description too, in agreeing words |
| vague names like `flag_2` or `val` | name it for a new colleague: the name is part of the prompt |
| asks for a total, or a date calculation | compute it in code |
| deep nesting, `extra="forbid"`, unions | flatten it |

**Your turn.** Make `Urgency` pass the checker: `reason` must be one of `money`, `delay`, `defect`,
`question`, and `explanation` at most 12 words, on four tickets.

In [27]:
class Urgency(BaseModel):
    reason: str          # <- one of: money, delay, defect, question
    explanation: str     # <- at most 12 words, and make sure the model knows it


def check_urgency():
    reasons = set()
    for tid in ("TCK-009", "TCK-003", "TCK-002", "TCK-018"):
        r = ask_schema(Urgency, as_block(by_id[tid]))
        if r.parsed is None:
            return f"{tid}: .parsed is None. Do your words agree with your limit?"
        if len(r.parsed.explanation.split()) > 12:
            return f"{tid}: the explanation has {len(r.parsed.explanation.split())} words"
        reasons.add(r.parsed.reason)
    if not reasons <= {"money", "delay", "defect", "question"}:
        return f"reasons outside the list: {reasons}"
    return f"passed: {sorted(reasons)}"


print(check_urgency())

TCK-009: the explanation has 39 words


## 3. Extraction: copy, then check

Triage **picks** from lists we wrote. Extraction **copies** values out of the customer's text: order
numbers, how long they've had the item, which product. There's no list to pick from, so `Literal`
can't help, and step 4 showed that `pattern` isn't enforced. The failure here is a plausible value that
isn't in the ticket at all.

<img src="figures/copy-and-check.png" width="900"
     alt="Extraction in four stages. The model copies an order number out of the ticket; a before
     validator normalises it, turning ord5004, 5002 and Bangla digits into ORD-NNNN; an after
     validator checks it against the ticket text passed in as validation context and rejects
     anything that isn't there; then code decides the return window from the copied facts. A side
     panel compares a required field, which invented an order number on 13 of 20 tickets, with a list
     that may be empty, which invented none.">

### 3.1 A required field invents

The first schema anyone writes for this:

In [28]:
class OrderRef(BaseModel):
    order_id: str = Field(description="The customer's order number, e.g. ORD-5001.")


wrong = []
for t in RAW_TICKETS:
    got = ask_schema(OrderRef, as_block(t)).parsed.order_id
    in_ticket = re.findall(r"ORD-\d+", t["body"])
    if got and got not in in_ticket:
        wrong.append((t["id"], got))

print(f"{len(wrong)} of 20 tickets came back with an order number that isn't in the ticket:")
for tid, got in wrong:
    print(f"  {tid}  {got}")

13 of 20 tickets came back with an order number that isn't in the ticket:
  TCK-002  ORD-5001
  TCK-006  ORD-5001
  TCK-007  REF-9001
  TCK-008  ORD-5001
  TCK-009  ORD-5001
  TCK-010  ORD-5001
  TCK-011  ORD-5001
  TCK-014  ORD-5001
  TCK-015  ORD-5001
  TCK-017  SW-88231
  TCK-018  ORD-5001
  TCK-019  null
  TCK-020  ORD-5001


Look at which number keeps coming back. `ORD-5001` is the example in our own description. Session 2's
section 7.3 showed a model copying an example's details into unrelated replies; here the example sits
inside the schema, with the same effect. A refund reference and a tracking number get promoted to
order numbers as well, and one ticket gets the string `"null"`.

The required `str` is the real cause. It leaves the model no honest way to say "there isn't one", so
it writes the most plausible thing it can. A `pattern=r"^ORD-\d{4}$"` wouldn't help either: Gemini
doesn't enforce it, and `ORD-5001` matches it anyway.

Why this matters: in session 5 the assistant looks order numbers up. An invented `ORD-5001` doesn't
fail. It returns Ayesha's order to somebody who isn't Ayesha.

### 3.2 Give it a way to say "none"

In [29]:
class Facts(BaseModel):
    order_ids: list[str] = Field(
        description="Every order number (ORD-NNNN) exactly as the customer wrote it. "
                    "Empty list if there is none. Never invent one.")


for t in RAW_TICKETS[:8]:
    print(t["id"], ask_schema(Facts, as_block(t)).parsed.order_ids)

TCK-001 ['ORD-5001']


TCK-002 []


TCK-003 ['ORD-5002']


TCK-004 ['ORD-5003']


TCK-005 ['ORD-5004']


TCK-006 []


TCK-007 []


TCK-008 []


Empty lists where there's nothing to copy. Over all twenty tickets this design invented nothing in my
runs, and `score.py` in section 4 counts it every time. But our inbox is tidy, and customers aren't:

In [30]:
messy = {
    "no prefix":     "My order number is 5002 I think, the kettles. Can I get a refund?",
    "typo":          "where is ord5004?? ordered 3 weeks ago",
    "Bangla digits": "আমার অর্ডার নম্বর ORD-৫০০৩, এখনো পাইনি।",
}
for name, body in messy.items():
    runs = [ask_schema(Facts, f"<ticket>\n{body}\n</ticket>").parsed.order_ids for _ in range(3)]
    print(f"{name:14} {runs}")

no prefix      [[], [], []]


typo           [['ORD-5004'], [], ['ORD-5004']]


Bangla digits  [['ORD-৫০০৩'], ['ORD-৫০০৩'], ['ORD-৫০০৩']]


Three different problems in nine calls. The bare `5002` was left out: the description says
`ORD-NNNN`, and a number without the prefix didn't qualify. (That's the safer way to fail. A missed
number costs a person one lookup; a bare number guessed to be an order could be a price.) The typo was
sometimes tidied into `ORD-5004` and sometimes missed. And the Bangla digits were copied faithfully,
so a lookup for `ORD-৫০০৩` finds nothing.

The model is following a format we described in words, and it does so differently from call to call.
Code can apply that format the same way every time. Two validators do it, and their order matters.

### 3.3 Normalise first

A `before` validator sees the raw list and rewrites every entry into one canonical form. Anything that
isn't shaped like an order number is rejected rather than converted, because `REF-9001` is not order
9001.

In [31]:
BANGLA_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
ORDER_NUMBER = re.compile(r"^(ORD)?[\s\-#]*(\d+)$", re.IGNORECASE)   # ORD-5002, ord5004, #5002, 5002


class FactsNormalised(Facts):
    @field_validator("order_ids", mode="before")
    @classmethod
    def normalise(cls, ids):
        clean = []
        for raw in ids:
            text = str(raw).strip().translate(BANGLA_DIGITS)
            if not text:
                continue                                  # an empty string means "none"
            match = ORDER_NUMBER.match(text)
            if match is None:
                raise ValueError(f"{raw!r} is not an order number")
            clean.append(f"ORD-{match.group(2)}")
        return clean


print(FactsNormalised(order_ids=["5002", "ord5004", "ORD-৫০০৩"]))
try:
    FactsNormalised(order_ids=["REF-9001"])
except ValidationError as err:
    print(err.errors()[0]["msg"])

order_ids=['ORD-5002', 'ORD-5004', 'ORD-5003']
Value error, 'REF-9001' is not an order number


`FactsNormalised` inherits the field from `Facts` and adds a validator, which is a handy way to build
up a model in steps. Every order number now has one form. But a tidy form proves nothing: an invented
`5001` normalises to a perfectly tidy `ORD-5001`.

### 3.4 Check it against the ticket

Every order number should appear in the ticket the model read. The validator needs the ticket text,
which isn't part of the data being validated. Pydantic's **validation context** exists for exactly this:
extra information you pass in alongside the data, which validators can read.

In [32]:
from pydantic import ValidationInfo


class CheckedFacts(FactsNormalised):
    @field_validator("order_ids")
    @classmethod
    def must_be_in_ticket(cls, ids, info: ValidationInfo):
        ticket = (info.context or {}).get("ticket")
        if ticket is None:                       # no context given: nothing to check against
            return ids
        text = ticket.translate(BANGLA_DIGITS)
        for order_id in ids:
            if order_id.removeprefix("ORD-") not in text:
                raise ValueError(f"{order_id} is not in the ticket")
        return ids


body = "where is ord5004?? ordered 3 weeks ago"
print(CheckedFacts.model_validate_json('{"order_ids": ["ord5004"]}', context={"ticket": body}))
try:
    CheckedFacts.model_validate_json('{"order_ids": ["ORD-5001"]}', context={"ticket": body})
except ValidationError as err:
    print(err.errors()[0]["msg"])

order_ids=['ORD-5004']
Value error, ORD-5001 is not in the ticket


The two validators run in sequence on the same field: `normalise` on the raw input, then
`must_be_in_ticket` on the clean list. Now run the check over every invented number from 3.1:

In [33]:
caught = 0
for tid, got in wrong:
    try:
        CheckedFacts.model_validate({"order_ids": [got]}, context={"ticket": by_id[tid]["body"]})
    except ValidationError:
        caught += 1

print(f"the check rejects {caught} of the {len(wrong)} invented order numbers")

the check rejects 13 of the 13 invented order numbers


This is session 2's `quote_is_real` (section 7.5), moved from a notebook check into the schema, where
it runs on every call.

Two practical catches, both about `.parsed`. When you pass a class as `response_schema`, the SDK runs
its validators itself to fill `.parsed`, and it has no context to give them. That's why the validator
above returns early when `info.context` is `None`: written as `info.context["ticket"]`, it would crash
inside the SDK call. And because `.parsed` never ran the real check, `ask_json` in `project/utils.py`
ignores `.parsed` and calls `TicketFacts.model_validate_json(r.text, context={"ticket": ...})` itself.

### 3.5 The model reads, code decides

Some facts arrive as words: "I've had them nine days", "got it on the 2nd, today is the 20th",
`১২ দিন আগে`. Turning those words into a number is reading, and models read well. Deciding what the
number means under the returns policy is a rule, and rules belong in code. v0.2's `TicketFacts` has
`days_since_delivery`, `product_kind` and `opened`, and `return_window()` in `project/policy.py` applies
the policy: 15 days for opened audio products, 30 for everything else, and `unknown` when the customer
didn't say.

In [34]:
from policy import return_window
from schemas import TicketFacts

samples = {
    "TCK-002":       by_id["TCK-002"]["body"],
    "relative date": "Got the PulseFit on the 2nd of this month, today is the 20th. Can I return it?",
    "Bangla":        "আমি ১২ দিন আগে হেডফোন পেয়েছি, খুলে ব্যবহার করেছি। এখন ফেরত দিতে চাই।",
}
for name, text in samples.items():
    r = ask_schema(TicketFacts, f"<ticket>\n{text}\n</ticket>")
    facts = TicketFacts.model_validate_json(r.text, context={"ticket": text})
    print(f"{name:14} days={facts.days_since_delivery}  kind={facts.product_kind}  "
          f"opened={facts.opened}  ->  {return_window(facts)}")

TCK-002        days=9  kind=audio  opened=True  ->  inside


relative date  days=18  kind=wearable  opened=None  ->  inside


Bangla         days=12  kind=audio  opened=True  ->  inside


The model did the reading; `return_window()` made the decision, in a few lines anyone can review.

These are still **claims**. The customer says nine days; the order record might say thirty. In session
5 the assistant checks the record, and this column becomes a fact instead of a hint. Until then it goes
in the spreadsheet for the person handling the ticket, not into the reply.

### 3.6 Images are input too

TCK-011 says the customer received the wrong item, and attaches a photo. An image is just another part
of `contents`, and the schema works the same way:

In [35]:
class DamageReport(BaseModel):
    product_kind: Literal["audio", "wearable", "kitchen", "bag", "other"]
    damage_visible: bool
    matches_complaint: bool = Field(description="Does the photo show what the customer describes?")


photo = (settings.DATA_DIR / "images" / "ticket-photo-headphones.jpg").read_bytes()
t11 = by_id["TCK-011"]
r = client.models.generate_content(
    model=MODEL,
    contents=[types.Part.from_bytes(data=photo, mime_type="image/jpeg"),
              as_block(t11) + "\nThe photo above is the one the customer attached."],
    config={"response_mime_type": "application/json", "response_schema": DamageReport})

print(t11["body"])
print(r.parsed)

I ordered the smartwatch and received a backpack. Photo attached. I need the watch, not this.
product_kind='audio' damage_visible=True matches_complaint=False


The customer says they received a backpack; the photo shows headphones. `matches_complaint=False` is
one `if` away from "send this ticket to a person".

**Your turn.** Refund references look like `REF-9001`. Write `MyFacts` so the checker passes: describe
the field, normalise `ref 9001` to `REF-9001`, and reject any reference that isn't in the ticket.
Remember to return early when there's no context, because `ask_schema` hands the class to the SDK.

In [36]:
class MyFacts(BaseModel):
    refund_refs: list[str]      # <- describe it, normalise it, check it against the ticket (context)


def check_refs():
    cases = {by_id["TCK-007"]["body"]: {"REF-9001"},
             "My refund ref is ref 9001 and still nothing has arrived.": {"REF-9001"},
             "Where is my refund? It's for order ORD-5002.": set()}
    for text, expected in cases.items():
        r = ask_schema(MyFacts, f"<ticket>\n{text}\n</ticket>")
        try:
            got = set(MyFacts.model_validate_json(r.text, context={"ticket": text}).refund_refs)
        except ValidationError as err:
            return f"a valid answer was rejected on {text[:40]!r}: {err.errors()[0]['msg']}"
        if got != expected:
            return f"{text[:40]!r}: expected {expected}, got {got}"
    return "passed"


print(check_refs())

'My refund ref is ref 9001 and still noth': expected {'REF-9001'}, got {'ref 9001'}


## 4. Legal, or right?

`Literal` guarantees the category is one of nine. It doesn't guarantee it's the right one. Every
ticket in `tickets.yml` has a category written by hand, so we can check.

Session 2's `grade()` ran a prompt three times and counted replies that passed a regex. That was a
smoke test, and session 2 said so. `score.py`, beside this notebook, is the grown-up version: it runs
the project's own code over all twenty tickets, compares triage with the hand labels, and counts the
order numbers `extract()` found against a regex over each ticket.

In [37]:
import subprocess

done = subprocess.run([sys.executable, "score.py"], capture_output=True, text=True, timeout=900)
print(done.stdout or done.stderr[-1500:])

triage: 18/20 correct · 1091 output tokens
  TCK-003  labelled complaint         model said order_status
  TCK-016  labelled refund            model said order_status

order numbers: 7 found · 0 missed · 0 invented · 0 ticket(s) rejected by the validator



Those are two different kinds of number:

| | Compared with | What it really measures |
|---|---|---|
| triage accuracy | the hand labels | agreement with **our** labelling convention. TCK-003 (an angry "where is my order") is labelled `complaint` and is defensibly `order_status`. A couple of points sit above a ceiling no prompt can lift |
| order numbers | a regex over the ticket text | nothing to argue about: a number is in the text or it isn't |

Report missed and invented separately, and never average them. A missed order number costs a person
one lookup; an invented one shows a customer somebody else's order.

### 4.1 Rules or examples?

Session 2 promised this. The guides recommend examples, and they taught us how to write them. Do they
help *this* task? `score.py --compare` scores three triage prompts, three runs each:

- **bare**: "You classify ShopWise customer support tickets."
- **rules**: `triage_prompt` from `prompts.yml`, which writes ShopWise's conventions down as four rules
- **rules + examples**: the same, plus three worked examples (written fresh, not taken from the inbox)

That's about 180 calls; it takes a few minutes, so run it before class or let it run while we talk.

In [38]:
done = subprocess.run([sys.executable, "score.py", "--compare"],
                      capture_output=True, text=True, timeout=1800)
print(done.stdout or done.stderr[-1500:])

bare instruction       [13, 14, 14] out of 20   (avg 1099 output tokens)
conventions as rules   [18, 18, 16] out of 20   (avg 1103 output tokens)
rules + 3 examples     [18, 17, 17] out of 20   (avg 1159 output tokens)



Writing the conventions down as rules moved the score clearly. The examples added nothing on top, and
they're paid for in input tokens on every call, so they ship switched off (`USE_FEW_SHOT=False` in
`config.py`). That's session 2's rule, *the guides are hypotheses and your measurement decides*, with
real data behind it. It doesn't mean examples never help: they teach shape (session 2, section 7.3).
Here the job is a labelling convention, and the rules already state it.

### 4.2 How big is the noise?

Look at the rules row again: the same prompt on the same twenty tickets scores differently from run to
run. Across all my runs it landed anywhere from 16 to 19. So:

- **A one- or two-ticket difference is noise.** Twenty tickets can tell a bad prompt from a good one;
  they can't tell two good prompts apart.
- **Always run the comparison more than once** before believing it. Session 9 builds a larger
  evaluation for this reason.
- **An example in your prompt is never a test case.** The first version of these examples reused three
  inbox tickets and scored a perfect 20/20, because three of the answers were sitting in the prompt.

## 5. Prompt caching: pay once for the part that doesn't change

Session 2 gave two pieces of advice about caching without testing them. Keep the system instruction
identical between calls, because providers cache a repeated beginning. And putting today's date first
costs one cache miss a day. Tonight we test both.

How caching works: the provider keeps the processed form of a prompt's beginning for a while. When the
next request starts with exactly the same tokens, that part is reused, which makes it cheaper (cached
tokens cost about a tenth of normal input tokens on Gemini) and faster. Only a shared **beginning**
counts. Change one token near the start and everything after it is new.

<img src="figures/prompt-cache.png" width="900"
     alt="Prompt caching. A long prompt drawn as a row of blocks: a stable prefix followed by a short
     question. On call 1 nothing is cached; on calls 2 to 5 most of the prefix is served from cache,
     about 20,000 of 25,000 tokens. A second panel shows the date at the start
     changing on a new day, causing one miss before hits resume. A third panel contrasts implicit
     caching, which is automatic and partial, with explicit caching, which stores the whole prefix for
     a lifetime you choose.">

### 5.1 Is our prompt cached?

v0.2's reply call sends the same system slot for every ticket. Does Gemini cache it?

In [39]:
from triage import reply_system

system = reply_system(date.today())
print("reply system slot:", client.models.count_tokens(model=MODEL, contents=system).total_tokens, "tokens\n")

for i in range(3):
    u = client.models.generate_content(
        model=MODEL, contents=as_block(by_id["TCK-001"]),
        config={"system_instruction": system}).usage_metadata
    print(f"call {i + 1}: input {u.prompt_token_count}, cached {u.cached_content_token_count}")

reply system slot: 684 tokens



call 1: input 741, cached None


call 2: input 741, cached None


call 3: input 741, cached None


`None`: nothing cached. Gemini's automatic ("implicit") caching only starts on a much longer prefix.
Google's docs give a 4,096-token minimum for the Flash models and don't list Flash-Lite; in my runs on
Flash-Lite nothing was cached below about 6,500 tokens, and above that it caches in blocks of roughly
4,000. Our prompt is far below that. Session 2's advice is still worth following, because it costs
nothing, but today it buys nothing either.

### 5.2 When caching pays

Caching pays when a long prefix is reused many times: a product manual every product question goes
through, a policy handbook, a contract, a long system prompt in a batch job. Our stand-in is the
library from session 2's section 7.2, ShopWise's returns policy hidden among sixty partner policies,
about 25,000 tokens. Five questions against it:

In [40]:
policy = Path("../session-02/returns_policy.md").read_text(encoding="utf-8")
partner = policy.replace("ShopWise", "Northwind").replace("15 days", "10 days").replace("30 days", "45 days")
docs = [f"<document index='{i}'>\n{partner}\n</document>" for i in range(60)]
docs.insert(37, f"<document index='shopwise'>\n{policy}\n</document>")
LIBRARY = "\n".join(docs)

QUESTIONS = ["How many days do I have to return opened headphones?",
             "How long does a refund take to arrive?",
             "Who pays for return shipping?",
             "Can I return a gift card?",
             "How do I start a return?"]


def ask_library(system, question):
    '''Ask one question with `system` as the system instruction; print what was cached.'''
    start = time.perf_counter()
    r = client.models.generate_content(
        model=MODEL, contents=f"Under ShopWise's policy: {question} Answer in one sentence.",
        config={"system_instruction": system})
    u = r.usage_metadata
    print(f"input {u.prompt_token_count:6,}  cached {u.cached_content_token_count or 0:6,}  "
          f"{time.perf_counter() - start:4.1f}s  {r.text.strip()[:55]}")


for q in QUESTIONS:
    ask_library(LIBRARY, q)

input 24,658  cached      0   1.2s  Under ShopWise's policy, you have 15 days from the deli


input 24,656  cached 20,450   1.2s  Approved refunds are paid to the original payment metho


input 24,653  cached 20,450   1.0s  Return shipping is paid by the customer, except for gol


input 24,654  cached 20,450   1.4s  No, gift cards and store credit cannot be returned unde


input 24,654  cached 20,450   1.3s  To start a return under ShopWise's policy, reply to you


The first call caches nothing; it's the one that fills the cache. From the second call on, about
20,000 of the 25,000 tokens come from the cache. Not all of them: implicit caching works in blocks
and promises nothing (Google's own words are "no cost saving guarantee"). Speed varies. In this run
the times barely moved; in others the cached calls were about a third faster. The billing saving is
the dependable part. It's automatic and needs no code. The one thing you control is order: stable content first, the part that changes last.
That's session 2's two-slot layout again.

### 5.3 What breaks a cache

Session 2's `reply.py` puts today's date at the very start of the system slot, because the model used
it more reliably there. What does that cost?

In [41]:
for day in ("2026-10-09", "2026-10-09", "2026-10-10", "2026-10-10"):
    print(day, end="  ")
    ask_library(f"Today's date is {day}.\n\n" + LIBRARY, QUESTIONS[0])

2026-10-09  

input 24,676  cached      0   1.2s  Under ShopWise's policy, you have 15 days from delivery
2026-10-09  

input 24,676  cached 20,450   1.3s  Under ShopWise's policy, you have 15 days from delivery
2026-10-10  

input 24,676  cached      0   1.2s  Under ShopWise's policy, you have **15 days of delivery
2026-10-10  

input 24,676  cached 20,450   1.1s  Under ShopWise's policy, you have 15 days from delivery


The first call with a new date misses, and the next one hits again. One miss a day, as session 2
claimed. The same applies to anything per-request at the top of a prompt: a customer's name, the
ticket itself, a timestamp with seconds in it. Put any of those first and nothing is ever cached.
Per-request content goes last.

### 5.4 Explicit caching 💳

Implicit caching is Gemini's decision. With **explicit** caching you create the cache yourself, give
it a lifetime (TTL), and refer to it by name. The whole prefix is cached, every time, and you pay a
storage fee for each hour it lives. On Flash-Lite, explicit caching isn't available on the free tier;
without billing, the cell says so and the saved output shows what it does.

In [42]:
cache = None
try:
    cache = client.caches.create(
        model=MODEL,
        config=types.CreateCachedContentConfig(system_instruction=LIBRARY, ttl="600s"))
    print(f"cache {cache.name}\n  {cache.usage_metadata.total_token_count:,} tokens, expires {cache.expire_time}\n")

    for q in QUESTIONS[:2]:
        r = client.models.generate_content(
            model=MODEL, contents=f"Under ShopWise's policy: {q} Answer in one sentence.",
            config={"cached_content": cache.name})
        u = r.usage_metadata
        print(f"input {u.prompt_token_count:,}  cached {u.cached_content_token_count:,}  {r.text.strip()[:55]}")
except errors.ClientError as err:
    print("This cell needs a billing-enabled key:", err.message[:120])
    print("With one, it caches all 25,000 tokens and every call reports them as cached.")

cache cachedContents/kidws2irwm4b57tmq13rv8lun6wvuv3o3ci0d4s6
  24,634 tokens, expires 2026-10-03 00:53:07.151709+00:00



input 24,658  cached 24,634  Under ShopWise's policy, you have 15 days from the deli


input 24,656  cached 24,634  Approved refunds are paid to the original payment metho


All of the prefix is cached, not a block of it. Two rules come with explicit caches. The system
instruction now lives inside the cache, so a request can't send another one:

In [43]:
if cache:
    try:
        client.models.generate_content(
            model=MODEL, contents="hello",
            config={"cached_content": cache.name, "system_instruction": "Be brief."})
    except errors.ClientError as err:
        print(err.code, err.message[:120])

    client.caches.delete(name=cache.name)        # and delete it when you're done: storage is billed hourly
    print("deleted")

400 CachedContent can not be used with GenerateContent request setting system_instruction, tools or tool_config.

Proposed f


deleted


Explicit caching earns its storage fee when you know the same prefix will be used many times within
the TTL: a batch job pushing 10,000 tickets through one long instruction, or a user asking many
questions about one document. Delete the cache when the job ends.

### 5.5 The other providers

Every major provider caches, and each does it differently. Read the provider's page before relying on
it, because the numbers change:

| | Gemini | OpenAI | Anthropic |
|---|---|---|---|
| How you turn it on | implicit: automatic. Explicit: `caches.create`, then `cached_content=` | automatic | mark the end of the cached part with `cache_control` (or one top-level `cache_control` for automatic mode) |
| Smallest prefix cached | 4,096 tokens on Flash (docs); about 6,500 on Flash-Lite in our runs | 1,024 tokens | 4,096 on Haiku 4.5; less on larger models |
| Price of cached tokens | about 10% of input; explicit caches add hourly storage | large discount on reads (up to 95%) | reads 10% of input; writing the cache costs 25% more than input |
| How long it lives | implicit: short. Explicit: the TTL you set (default 1 hour) | minutes, or up to 24 hours | 5 minutes, or 1 hour |
| Usage field | `cached_content_token_count` | `usage.input_tokens_details.cached_tokens` | `cache_read_input_tokens`, `cache_creation_input_tokens` |
| Docs | [Gemini caching](https://ai.google.dev/gemini-api/docs/generate-content/caching) | [OpenAI prompt caching](https://developers.openai.com/api/docs/guides/prompt-caching) | [Anthropic prompt caching](https://platform.claude.com/docs/en/build-with-claude/prompt-caching) |

The same library on the other two, if you have the keys:

In [44]:
OPENAI_MODEL = "gpt-5.4-nano"                    # 🔑
ANTHROPIC_MODEL = "claude-haiku-4-5-20251001"    # 🔑

if os.getenv("OPENAI_API_KEY"):
    import openai
    for i in range(2):
        r = openai.OpenAI().responses.create(model=OPENAI_MODEL, instructions=LIBRARY, input=QUESTIONS[0])
        print(f"OpenAI call {i + 1}: {r.usage.input_tokens_details.cached_tokens:,} of "
              f"{r.usage.input_tokens:,} input tokens cached")
else:
    print("🔑 no OPENAI_API_KEY. Saved run: 0, then 22,272 of 23,324 tokens cached, with no code change.")

if os.getenv("ANTHROPIC_API_KEY"):
    import anthropic
    for i in range(2):
        m = anthropic.Anthropic().messages.create(
            model=ANTHROPIC_MODEL, max_tokens=60,
            system=[{"type": "text", "text": LIBRARY, "cache_control": {"type": "ephemeral"}}],
            messages=[{"role": "user", "content": QUESTIONS[0]}])
        print(f"Anthropic call {i + 1}: wrote {m.usage.cache_creation_input_tokens:,}, "
              f"read {m.usage.cache_read_input_tokens:,} from cache")
else:
    print("🔑 no ANTHROPIC_API_KEY. Saved run: wrote 26,232 tokens, then read 26,232.")

OpenAI call 1: 0 of 23,324 input tokens cached


OpenAI call 2: 22,272 of 23,324 input tokens cached


Anthropic call 1: wrote 26,232, read 0 from cache


Anthropic call 2: wrote 0, read 26,232 from cache


OpenAI needed no code at all. Anthropic needed one marker saying where the cached part ends, and
charged extra to write it. From session 4 you'll reach all three through LangChain, which passes each
provider's own mechanism through (`cached_content` for Gemini, `cache_control` for Anthropic, nothing
for OpenAI). The mechanism stays the provider's, which is why it's worth knowing before the framework
hides it.

### 5.6 When to cache

| Cache when the prefix is... | Don't bother when... |
|---|---|
| long: thousands of tokens | the prompt is short (v0.2 today) |
| byte-for-byte identical on every call | it starts with something per-user or per-request |
| reused many times within minutes (implicit) or within your TTL (explicit) | calls are rare, so the cache expires between them |

And when a document is too big for the context window, or only a small part of it matters for each
question, don't send all of it, cached or not. Find the relevant part and send only that. That's
retrieval, and it's session 8.

## 6. Project v0.2

<img src="figures/v02-pipeline.png" width="880"
     alt="Version 0.2's pipeline. The inbox file is validated into Ticket objects. Each ticket goes to
     triage, which returns a TriageResult whose money rule sets needs_human, and to extract, which
     returns TicketFacts whose order numbers are normalised and checked against the ticket. Code
     decides the return window from the facts. The triage steers the drafted reply, and every ticket
     becomes one row of triage.csv.">

We walk through this in the editor and the terminal, not here. Open `session-03/project/`:

```
session-03/
├── score.py          section 4's measurements. Not part of the assistant
└── project/
    ├── config.py     ★ Settings(BaseSettings): typed configuration (1.7)
    ├── schemas.py    ★ Ticket, TriageResult, TicketFacts and their validators (1, 2, 3)
    ├── policy.py     ★ return_window(): the return rules, in code (3.5)
    ├── utils.py      ★ build_client(), ask_json(), load_inbox(): the plumbing
    ├── prompts.yml     v0.1's prompts + triage_prompt, few_shot_examples (off), facts_prompt
    └── triage.py     ★ the program: replaces v0.1's reply.py
```

| File | Why it's a separate file |
|---|---|
| `config.py` | every setting, typed and defaulted, in one list. `uv run python config.py` prints what it resolved to |
| `schemas.py` | the shapes, in one place. Session 10 imports `TriageResult` unchanged; session 13 returns it from an API |
| `policy.py` | business rules a manager might ask to change, kept apart from the code that talks to the model |
| `utils.py` | what every LLM program needs, so `triage.py` reads as the program and nothing else |
| `score.py` | beside `project/`, not in it: it measures the assistant, and later snapshots don't need a copy |

Where tonight's sections went:

| In the project | From section |
|---|---|
| `load_inbox()`: `TypeAdapter(list[Ticket])`, so a bad fixture fails at startup | 1.2, 1.6 |
| `Settings`: the `SHOPWISE_` prefix, `SecretStr`, a real `bool` | 1.7 |
| `TriageResult`: `Literal` fields, descriptions in agreeing words, decisions first, the money rule | 1.5, 2 |
| `ask_json()`: validates `r.text` itself (never `.parsed`), passes context, repairs once | 2 step 4, 3.4 |
| `TicketFacts`: a list that may be empty, normalised before, checked against the ticket after | 3.2 to 3.4 |
| `return_window()` | 3.5 |
| `USE_FEW_SHOT=False`, with the measurement behind it | 4.1 |
| `build_client()` with `HttpRetryOptions`. **v0.1's `with_retry` is deleted**, as session 2 promised | setup |

v0.2 no longer streams replies. v0.1 streamed so a customer could watch the reply arrive; v0.2 is a
batch job over the inbox, and nobody is watching. Streaming returns in session 6, in the chat.

From a terminal in `session-03/project/`:

```bash
uv run python triage.py --limit 3                       # three tickets, with drafted replies
uv run python triage.py --no-reply --csv ../triage.csv  # the whole inbox, as the CTO's spreadsheet
```

The next two cells run the same commands so the output is saved for anyone without a key.

In [45]:
done = subprocess.run([sys.executable, "triage.py", "--limit", "3"],
                      cwd="project", capture_output=True, text=True, timeout=600)
print(done.stdout[-4000:] or done.stderr[-1500:])

ShopWise Support Assistant v0.2.0 · gemini-3.5-flash-lite · 3 ticket(s)

TCK-001  order_status | medium | calm | needs_human=False
  Customer is asking for tracking update on order ORD-5001.
  orders=['ORD-5001']  days=None  return window: unknown
----------------------------------------------------------------------------
Hello,

I'm sorry your headphones haven't arrived yet. A tracking page that stops moving is frustrating, especially when you are waiting on your order.

I cannot see delivery tracking or change shipping dates from my side. I have passed your message to a colleague on our delivery team so they can look into it. They will reply to this email with an update.

ShopWise Support

TCK-002  complaint | high | frustrated | needs_human=True
  Customer reports a crackling earcup after nine days and requests a refund.
  orders=-  days=9  return window: inside
----------------------------------------------------------------------------
Hello,

I'm sorry your headphones are crackl

In [46]:
done = subprocess.run([sys.executable, "triage.py", "--no-reply", "--csv", "../triage.csv"],
                      cwd="project", capture_output=True, text=True, timeout=900)
print(done.stdout.splitlines()[-1] if done.stdout else done.stderr[-1500:])

import csv

with open("triage.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
for row in rows:
    print(f"{row['id']}  {row['category']:16} {row['urgency']:6} human={row['needs_human']:5} "
          f"orders={row['order_ids'] or '-':9} window={row['return_window']}")

wrote 20 rows to ../triage.csv
TCK-001  order_status     medium human=False orders=ORD-5001  window=unknown
TCK-002  complaint        high   human=True  orders=-         window=inside
TCK-003  order_status     high   human=True  orders=ORD-5002  window=unknown
TCK-004  order_status     medium human=False orders=ORD-5003  window=unknown
TCK-005  warranty         low    human=True  orders=ORD-5004  window=unknown
TCK-006  warranty         low    human=True  orders=-         window=unknown
TCK-007  refund           low    human=True  orders=-         window=unknown
TCK-008  product_question low    human=False orders=-         window=unknown
TCK-009  billing          high   human=True  orders=-         window=unknown
TCK-010  product_question low    human=False orders=-         window=unknown
TCK-011  complaint        high   human=True  orders=-         window=unknown
TCK-012  shipping         medium human=False orders=ORD-5002  window=unknown
TCK-013  billing          low    human=True  o

That's the spreadsheet: one validated row per ticket, sortable, countable, and safe to `if` on.

What v0.2 still can't do, each a later session:

- **The facts are the customer's claims.** It can't look up an order or check a delivery date.
  Session 5.
- **Every line is Gemini-shaped**: the client, the config, the error classes, the schema dialect.
  Session 4.
- **Most guardrails are still instructions.** One is a validator now; the rest are sentences.
  Session 7.

## 7. Wrap-up

| v0.1 | v0.2 |
|---|---|
| Prose only | A validated `TriageResult` and `TicketFacts` for every ticket, and a CSV |
| Settings were `os.getenv` constants | `Settings`: typed, and fails at startup naming the field |
| The fixture was trusted | `tickets.yml` is validated when it loads |
| Guardrails lived in the prompt | "Money goes to a person" is a validator; order numbers are checked against the ticket |
| A hand-written retry for 429s | The SDK's retries, plus a repair loop for answers that fail validation |
| Pass counts from three runs | Accuracy over 20 labelled tickets, extraction counts, and the noise floor |
| "Keep the prefix stable" was advice | Measured: when caching starts, what it saves, what breaks it |

### Next session

> You show the CTO the spreadsheet. *"Good. Leadership wants a Plan B vendor by Friday, in case Google
> raises prices. How long to switch?"*

Every call in `triage.py` is `client.models.generate_content`. Every schema goes in through
`response_schema`, and every error class is Google's. Worse, section 2's last surprise has a mirror
image: Gemini rejects a schema containing `additionalProperties: false`, and OpenAI's strict mode
rejects a schema *without* it (appendix D shows both). There's no schema that satisfies both as
written, so switching vendors means rewriting all of it. Session 4 is one interface over all of them.

**Before then:** work through `exercises.ipynb`. The three-star ones put the session's two ideas
together: code that decides the queue, and finding where caching starts on your own key.

---

# Appendix

Optional, and not needed for session 4.

## A. `Enum` or `Literal`?

Both close the value space and produce the same JSON Schema. They differ in what the rest of your
program can do with the value.

| | `Literal["low", "medium", "high"]` | `class Urgency(str, Enum)` |
|---|---|---|
| the value is | a plain `str` | an enum member, `Urgency.HIGH` |
| renaming a value | search for a string | your editor renames it |
| attaching behaviour | nowhere to put it | methods on the enum |
| on the wire | `enum: [...]` | `enum: [...]`, identical |

This course uses `Literal` because it's one line and you can see the whole value space where it's
used. Reach for an `Enum` when behaviour belongs with the values, such as an SLA per urgency:

In [47]:
from enum import Enum


class Urgency(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"

    @property
    def sla_hours(self) -> int:
        return {"low": 72, "medium": 24, "high": 4}[self.value]


class Escalation(BaseModel):
    urgency: Urgency


e = Escalation(urgency="high")
print(e.urgency, "-> answer within", e.urgency.sla_hours, "hours")
print("on the wire:", Escalation.model_json_schema()["$defs"]["Urgency"]["enum"])

Urgency.HIGH -> answer within 4 hours
on the wire: ['low', 'medium', 'high']


## B. Raw JSON Schema, without Pydantic

`response_json_schema` takes a plain dict, which is useful when schemas are generated at runtime (from
a database table, or a form a user designed):

```python
config = {"response_mime_type": "application/json",
          "response_json_schema": {
              "type": "object",
              "properties": {"category": {"type": "string", "enum": ["billing", "shipping"]}},
              "required": ["category"]}}
```

You get back a dict and lose validation on the way in. That's the case for Pydantic in one sentence:
the same schema, plus the check.

## C. Does a reasoning field improve accuracy?

A popular tip: declare a `reasoning` field before `category`, so the model thinks before it decides.
Step 3 showed the mechanism is real, because fields really are written in order. Whether it helps is a
separate question. `score.py --schemas` scores three shapes on the inbox. The third is the control: the
same reasoning field, declared *after* the category, so it costs the same tokens but can't influence
the answer. Without a control you can't tell "the reasoning helped" from "a longer answer helped".

```bash
uv run python score.py --schemas
```

My run: category only 18/20, reasoning first 17/20, reasoning last 17/20, with the reasoning field
costing about six times the output tokens. No gain here, where the misses are convention
disagreements. Reasoning earns its place when a task needs several steps; session 5 measures it where
the reasoning changes which action is taken.

## D. The same schema on OpenAI and Anthropic 🔑

Every major provider supports structured output, and none spells it the same way.

In [48]:
if os.getenv("OPENAI_API_KEY"):
    import openai
    oc = openai.OpenAI()

    # The SDK's helper rewrites the Pydantic schema into OpenAI's strict form for you.
    r = oc.responses.parse(model=OPENAI_MODEL, instructions="Triage this ShopWise ticket.",
                           input=as_block(t9), text_format=TriageResult)
    print("OpenAI   ", r.output_parsed)

    # Send the plain Pydantic schema instead, the one Gemini accepted, and strict mode refuses it.
    try:
        oc.responses.create(model=OPENAI_MODEL, input=as_block(t9), text={"format": {
            "type": "json_schema", "name": "triage", "strict": True,
            "schema": TriageResult.model_json_schema()}})
    except openai.BadRequestError as err:
        print("          raw schema refused:", err.body["message"][:105])
else:
    print("🔑 no OPENAI_API_KEY. Saved run: parse() works; the raw Pydantic schema is refused with "
          "\"'additionalProperties' is required to be supplied and to be false\".")

if os.getenv("ANTHROPIC_API_KEY"):
    import anthropic
    m = anthropic.Anthropic().messages.parse(
        model=ANTHROPIC_MODEL, max_tokens=400, output_format=TriageResult,
        messages=[{"role": "user", "content": "Triage this ShopWise ticket.\n" + as_block(t9)}])
    print("Anthropic", m.parsed_output)
else:
    print("🔑 no ANTHROPIC_API_KEY. Saved run: messages.parse() returns a TriageResult.")

OpenAI    category='refund' urgency='high' sentiment='frustrated' needs_human=True summary='Customer reports two same-day $129 charges for one order; requests duplicate refund.'


          raw schema refused: Invalid schema for response_format 'triage': In context=(), 'additionalProperties' is required to be supp


Anthropic category='billing' urgency='high' sentiment='frustrated' needs_human=True summary='Customer charged twice for single order, requesting refund'


| | Gemini | OpenAI | Anthropic |
|---|---|---|---|
| how you ask | `response_schema=` in the config | `text_format=` (helper) or a strict `json_schema` | `output_format=` |
| what you read | `.parsed` | `.output_parsed` | `.parsed_output` |
| `additionalProperties: false` | **rejected** (400) | **required** in strict mode (400 without it) | accepted |

Gemini rejects the schema Pydantic's `extra="forbid"` produces, and OpenAI's strict mode demands it.
That contradiction is why switching providers is never a matter of changing a URL, and it's where
session 4 starts.

## E. Two more Pydantic tools: `computed_field` and `model_copy`

You'll meet both when session 10 passes state between graph steps.

In [49]:
from pydantic import computed_field


class Row(BaseModel):
    id: str
    received: date

    @computed_field          # derived from other fields; included when you dump the model
    @property
    def age_days(self) -> int:
        return (date(2026, 10, 3) - self.received).days


row = Row(id="TCK-001", received="2026-08-02")
print(row.model_dump())

later = row.model_copy(update={"id": "TCK-001-b"})     # a changed copy; the original is untouched
print(row.id, "->", later.id)

{'id': 'TCK-001', 'received': datetime.date(2026, 8, 2), 'age_days': 62}
TCK-001 -> TCK-001-b
